# Phase 1: Corpus Statistics and Quality Inspection

This notebook provides reproducible exploration and verification of the crawled Wikipedia corpus for the **Semantic Search Engine: TF-IDF vs Embedding-based Retrieval** project.

### Project Rules Checked:
1. Config-driven with fixed parameters.
2. Metrics generated programmatically from corpus files (no hardcoded or invented values).
3. Quality filters verified: minimum 100 words, English language filter, boilerplate stripped.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 120

corpus_path = Path("../data/raw/documents.jsonl")
stats_path = Path("../reports/results/corpus_stats.json")
state_path = Path("../data/raw/crawler_state.json")

## 1. Load Corpus and Inspect Document Schema

In [ ]:
docs = []
with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            docs.append(json.loads(line))

df = pd.DataFrame(docs)
print(f"Loaded {len(df):,} crawled documents.")
df[["doc_id", "title", "url", "word_count", "language", "crawl_timestamp"]].head()

## 2. Load Programmatic Corpus Statistics JSON

In [ ]:
if stats_path.exists():
    with open(stats_path, "r", encoding="utf-8") as f:
        stats = json.load(f)
    print("Corpus Statistics:")
    print(f"  - Document count: {stats['document_count']}")
    print(f"  - Total words: {stats['total_word_count']:,}")
    print(f"  - Vocabulary size: {stats['vocabulary_size']:,}")
    print(f"  - Mean length: {stats['length_distribution']['mean']} ± {stats['length_distribution']['std']} words")
    print(f"  - Median length: {stats['length_distribution']['median']} words")
    print(f"  - Duplicate rate: {stats['duplicate_rate']*100:.2f}%")
else:
    print("Run src/crawler/corpus_stats.py to generate stats JSON.")

## 3. Document Length Distribution (Word Count)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["word_count"], bins=30, kde=True, color="#2563EB")
plt.axvline(df["word_count"].median(), color="red", linestyle="--", label=f"Median: {df['word_count'].median():.0f}")
plt.axvline(df["word_count"].mean(), color="green", linestyle="-", label=f"Mean: {df['word_count'].mean():.1f}")
plt.title("Document Length Distribution (Words per Article)", fontsize=14, fontweight="bold")
plt.xlabel("Word Count")
plt.ylabel("Document Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Top Frequent Terms (Stopwords Filtered)

In [ ]:
if stats_path.exists() and "top_terms" in stats:
    top_df = pd.DataFrame(stats["top_terms"][:20])
    plt.figure(figsize=(12, 6))
    sns.barplot(data=top_df, x="count", y="term", palette="Blues_r")
    plt.title("Top 20 Content Terms in Corpus (Excluding Stopwords)", fontsize=14, fontweight="bold")
    plt.xlabel("Frequency")
    plt.ylabel("Term")
    plt.tight_layout()
    plt.show()

## 5. Sample Document Text and Headings Inspection

In [ ]:
sample_doc = docs[0]
print(f"Title: {sample_doc['title']}")
print(f"URL: {sample_doc['url']}")
print(f"Word count: {sample_doc['word_count']}")
print(f"Headings ({len(sample_doc['headings'])} total):")
for h in sample_doc["headings"][:8]:
    print(f"  [{h['level'].upper()}] {h['text']}")
print("\nBody snippet (first 400 chars):")
print(sample_doc["clean_body_text"][:400] + "...")